# `infra/`: seeds, configs, metrics, evaluation harness, prediction files and run records (walkthrough)

**Owner:** Aya, ML Infrastructure and Evaluation Engineer. **Branch:** `team/aya-ml-infrastructure-evaluation`.

This notebook explains the scripts in `infra/`, the small Python package every training role imports so that all experiments are configured, seeded, scored, saved and recorded **the same way**. It follows the formats and rules Hazem defined (the shared config, the example config, the four-class `customer_state` score with the `unclear` exception, the prediction file with forced probabilities, and the run record). It adds no new rules of its own to them.

**What this notebook does not do.** It trains no real model, reads no dataset CSVs or transcripts, and writes nothing inside the repo. Everything it saves goes to a temporary folder.

### How to read the numbers

| Symbol | Meaning |
|---|---|
| ✅ | A real value, read from `shared_config.json` or `final/class_weights.json` when you run the notebook. |
| 📋 | An example value copied from the project documents (Hazem's example config, the model selection report). |
| 🧪 | **Synthetic.** Made up only to show a format or check a calculation by hand. It is **not** a result of any model. Never quote it. |

### Contents
1. Setup
2. The shared config
3. Seeds
4. Mean and standard deviation across seeds
5. The experiment config
6. Per-field metrics (`metrics.py`)
7. The evaluation harness and mock predictions (`harness.py`, `mock_predictions.py`)
8. The predictions file (`predictions_file.py`)
9. The run record (`run_record.py`, schema version 2)
10. A template for your own experiment
11. Rules, and what is still to come

## 1. Setup

The scripts live in `infra/` at the repo root. To use them from any notebook, the repo root must be on `sys.path`. The cell below finds the repo root the same way Hazem's training skeleton does (the nearest parent folder that has a `.git`), and adds it to `sys.path`.

It also prints the Python version and the package versions. The team uses **Python 3.11** (see `requirements/README.md`). `torch` is optional for `infra/`: Aya's environment does not install it, and the code works with or without it.

In [1]:
import importlib.metadata
import json
import platform
import sys
import tempfile
from pathlib import Path


def find_repo_root(start):
    p = Path(start).resolve()
    for parent in [p, *p.parents]:
        if (parent / ".git").exists():
            return parent
    raise FileNotFoundError("Repo root not found. Run this notebook from inside the repo.")


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Everything this notebook saves goes into one temporary folder, never into the repo.
_TMP = tempfile.TemporaryDirectory()
WORK = Path(_TMP.name)


def fresh_dir():
    """A new empty temporary folder. Re-running a cell never hits 'file already exists'."""
    return Path(tempfile.mkdtemp(dir=WORK))


print("Python      :", platform.python_version())
print("Repo folder :", REPO_ROOT.name)
for name in ("numpy", "scikit-learn", "pandas", "pytest", "torch"):
    try:
        print(f"{name:<12}:", importlib.metadata.version(name))
    except importlib.metadata.PackageNotFoundError:
        print(f"{name:<12}: not installed")

Python      : 3.11.9
Repo folder : svr
numpy       : 1.26.4
scikit-learn: 1.9.1
pandas      : 2.1.4
pytest      : 8.3.4
torch       : not installed


In [2]:
from dataclasses import replace

import numpy as np

from infra.experiment_config import (
    ExperimentConfig,
    ExperimentConfigError,
    changed_fields,
    load_experiment_config,
    save_experiment_config,
)
from infra.harness import (
    DEFAULT_LABEL_STATUS,
    HarnessError,
    evaluate_predictions,
    load_truth_table,
)
from infra.metrics import (
    MetricsError,
    combine_macro_f1,
    confusion_matrix,
    evaluate_field,
    seed_metrics,
)
from infra.mock_predictions import make_mock_predictions
from infra.predictions_file import (
    PredictionsFileError,
    predictions_path,
    read_predictions_file,
    write_predictions_file,
)
from infra.run_record import (
    RunRecordError,
    build_run_record,
    load_run_record,
    make_seed_result,
    save_run_record,
)
from infra.seeds import run_over_seeds, set_global_seed, summarize, summarize_metrics
from infra.shared_config import SharedConfigError, load_shared_config

print("infra imported OK")

infra imported OK


## 2. The shared config (defined by Hazem)

`notebooks/training/shared_config.json` holds the values the whole team shares: the split seed, the five experiment seeds, the maximum sequence length, the dataset version and the row counts.

**Rule: `infra/` only reads this file. It never writes it and never keeps its own copy of the values.** If Hazem changes a value, everyone gets the change on the next run. That is why the cells below read the numbers from the file instead of typing them.

`load_shared_config` also **validates** the file, so a wrong value stops the run immediately instead of silently changing results.

In [3]:
CANDIDATES = [
    REPO_ROOT / "notebooks" / "training" / "shared_config.json",
    REPO_ROOT.parent / "svr-data" / "notebooks" / "training" / "shared_config.json",
    REPO_ROOT.parent / "smart-voicemail-router" / "notebooks" / "training" / "shared_config.json",
]
WHERE = ["inside this repo", "sibling folder svr-data", "sibling folder smart-voicemail-router"]

found = [(label, p) for label, p in zip(WHERE, CANDIDATES) if p.is_file()]
if not found:
    raise FileNotFoundError(
        "shared_config.json not found. Looked in: " + "; ".join(WHERE)
        + ". It lives at notebooks/training/shared_config.json on the data branch."
    )
where, SHARED_CONFIG_PATH = found[0]
shared = load_shared_config(SHARED_CONFIG_PATH)

print("shared_config.json found:", where)
print("loaded and validated OK  ✅")
print()
for key, value in shared.items():
    print(f"  {key:<17}{value!r}")

shared_config.json found: sibling folder svr-data
loaded and validated OK  ✅

  split_seed       42
  experiment_seeds [42, 1337, 2023, 7, 99]
  max_seq_len      128
  split_ratio      '70/20/10'
  dataset_version  'v2'
  dataset          'svr-voicemail-corpus'
  train_rows       3575
  val_rows         1003
  test_rows        518


**Checking the numbers against each other.** The config states the split ratio as text and the row counts as numbers. The cell below computes the percentages from the row counts and compares them with the stated ratio. Nothing is typed by hand.

In [4]:
rows = {s: shared[f"{s}_rows"] for s in ("train", "val", "test")}
total = sum(rows.values())
print(f"total clean rows ✅ : {total}")
print()
print(f"{'split':<7}{'rows':>7}{'share':>10}")
for split, n in rows.items():
    print(f"{split:<7}{n:>7}{100 * n / total:>9.2f}%")

try:
    stated = [int(x) for x in shared["split_ratio"].split("/")]
    rounded = [round(100 * n / total) for n in rows.values()]
    print()
    print("split_ratio in the config      :", shared["split_ratio"])
    print("rounded from the row counts    :", "/".join(map(str, rounded)))
    print("they agree                     :", stated == rounded)
except ValueError:
    print("split_ratio is not in the form 'a/b/c', so it was not compared.")

total clean rows ✅ : 5096

split     rows     share
train     3575    70.15%
val       1003    19.68%
test       518    10.16%

split_ratio in the config      : 70/20/10
rounded from the row counts    : 70/20/10
they agree                     : True


**What the loader rejects.** The cell below builds a few broken copies of the real config (in memory, in a temporary folder) and shows what the loader says about each. The first case is an untouched copy and must be accepted.

In [5]:
def try_load(label, data):
    path = fresh_dir() / "shared_config.json"
    path.write_text(json.dumps(data), encoding="utf-8")
    try:
        load_shared_config(path)
        print(f"{label:<22}: accepted")
    except SharedConfigError as exc:
        print(f"{label:<22}: REJECTED -> {exc}")


broken = {
    "untouched copy": dict(shared),
    "missing test_rows": {k: v for k, v in shared.items() if k != "test_rows"},
    "duplicate seeds": dict(shared, experiment_seeds=[42, 42, 7]),
    "a seed as text": dict(shared, experiment_seeds=[42, "1337"]),
    "split_seed = True": dict(shared, split_seed=True),
}
for label, data in broken.items():
    try_load(label, data)

untouched copy        : accepted
missing test_rows     : REJECTED -> shared config is missing keys: ['test_rows']
duplicate seeds       : REJECTED -> experiment_seeds has duplicates: [42, 42, 7]
a seed as text        : REJECTED -> experiment_seeds must be a non-empty list of integers, got [42, '1337']
split_seed = True     : REJECTED -> split_seed must be an integer, got True


## 3. Seeds

**Why five seeds?** Fine-tuning a model on a small dataset is noisy: the same setup with a different seed gives a different score, and the rare classes (for example `at_risk` churn) vary the most. One run can be lucky. The roles guide therefore requires every experiment to run on **all five seeds** and to report the mean and the standard deviation (see also `research/notes/training-stability-mosbach.md`).

**Two different jobs, easy to mix up:**
- `split_seed` decides how the data was split into train, validation and test. It is fixed. The data is never re-split.
- `experiment_seeds` are the seeds for training runs. Every experiment runs once per seed.

The number 42 appears in both, but the two have nothing to do with each other.

In [6]:
seeds = shared["experiment_seeds"]
print("split_seed       ✅ :", shared["split_seed"], "(used once, to split the data)")
print("experiment_seeds ✅ :", seeds, "(one training run per seed)")

split_seed       ✅ : 42 (used once, to split the data)
experiment_seeds ✅ : [42, 1337, 2023, 7, 99] (one training run per seed)


**`set_global_seed(seed)`** seeds Python's `random`, NumPy and, if it is installed, PyTorch. The same seed must give the same numbers, and a different seed a different one.

In [7]:
import random


def draw():
    return round(random.random(), 6), round(float(np.random.rand()), 6)


s_a, s_b = seeds[0], seeds[1]
set_global_seed(s_a); first = draw()
set_global_seed(s_a); again = draw()
set_global_seed(s_b); other = draw()

print(f"seed {s_a:<5}:", first)
print(f"seed {s_a:<5}:", again, "  (same seed again)")
print(f"seed {s_b:<5}:", other)
print()
print("same seed gives the same numbers      :", first == again)
print("a different seed gives other numbers  :", first != other)

seed 42   : (0.639427, 0.37454)
seed 42   : (0.639427, 0.37454)   (same seed again)
seed 1337 : (0.617753, 0.262025)

same seed gives the same numbers      : True
a different seed gives other numbers  : True


**`run_over_seeds(fn, seeds)`** calls `fn(seed)` once per seed, in order, and sets the global seed before each call. The function you pass is where your training and validation code goes. Below it is a toy function that only draws a random number 🧪, to show that running everything twice gives identical results.

Note: setting the global seed is not enough on its own. Whenever a library accepts a seed argument (for example `random_state=seed` in scikit-learn), pass the seed explicitly too.

In [8]:
def toy(seed):
    return round(float(np.random.rand()), 4)


run_1 = run_over_seeds(toy, seeds)
run_2 = run_over_seeds(toy, seeds)

print("🧪 toy function, one result per seed")
for rec in run_1:
    print(f"  seed {rec['seed']:<5} -> {rec['result']}")
print()
print("a second full run gives identical results:", run_1 == run_2)

🧪 toy function, one result per seed
  seed 42    -> 0.3745
  seed 1337  -> 0.262
  seed 2023  -> 0.322
  seed 7     -> 0.0763
  seed 99    -> 0.6723

a second full run gives identical results: True


**PyTorch is optional.** `infra/` must work on machines without `torch`, so it seeds `torch` only when it is installed. Lama's and Sira's environments have it; Aya's does not.

In [9]:
try:
    import torch
except ImportError:
    print("torch is not installed in this environment.")
    print("set_global_seed() skips it, and everything else works.")
else:
    set_global_seed(seeds[-1])
    print(f"torch is installed. After set_global_seed({seeds[-1]}):")
    print("  torch.initial_seed() =", torch.initial_seed())

torch is not installed in this environment.
set_global_seed() skips it, and everything else works.


## 4. Mean and standard deviation across seeds

For each metric, an experiment reports the **mean** and the **standard deviation (std)** over its five seed results. Hazem confirmed that the std is the **sample** standard deviation (`ddof=1`): it divides by `n - 1` (4 for five seeds), not by `n`. This is the right choice for a small sample.

The cell below computes it step by step on five made-up scores 🧪, then checks the result against NumPy and Python's `statistics` module.

In [10]:
import statistics

scores = [0.62, 0.65, 0.60, 0.64, 0.63]  # 🧪 synthetic
n = len(scores)
mean = sum(scores) / n

print(f"n = {n}, mean = sum / n = {mean:.4f}")
print()
print(f"{'score':>7}{'deviation':>12}{'squared':>12}")
squares = []
for s in scores:
    d = s - mean
    squares.append(d * d)
    print(f"{s:>7.2f}{d:>12.4f}{d * d:>12.6f}")

sum_sq = sum(squares)
variance = sum_sq / (n - 1)
std_by_hand = variance ** 0.5
print()
print(f"sum of squares          = {sum_sq:.6f}")
print(f"divide by n - 1 = {n - 1}     = {variance:.6f}   (the sample variance)")
print(f"square root             = {std_by_hand:.6f}   (the sample std)")
print()
ours = summarize(scores, ddof=1)
print("infra.summarize          :", round(ours["std"], 6))
print("numpy, ddof=1            :", round(float(np.std(scores, ddof=1)), 6))
print("statistics.stdev         :", round(statistics.stdev(scores), 6))
print("all four agree           :", abs(std_by_hand - ours["std"]) < 1e-12
      and abs(ours["std"] - statistics.stdev(scores)) < 1e-12)

n = 5, mean = sum / n = 0.6280

  score   deviation     squared
   0.62     -0.0080    0.000064
   0.65      0.0220    0.000484
   0.60     -0.0280    0.000784
   0.64      0.0120    0.000144
   0.63      0.0020    0.000004

sum of squares          = 0.001480
divide by n - 1 = 4     = 0.000370   (the sample variance)
square root             = 0.019235   (the sample std)

infra.summarize          : 0.019235
numpy, ddof=1            : 0.019235
statistics.stdev         : 0.019235
all four agree           : True


**`ddof=1` versus `ddof=0`.** With `ddof=0` the sum of squares is divided by `n` instead of `n - 1`, which gives a smaller number. With five seeds the two differ by a fixed factor, `sqrt(n / (n - 1))`. Every result records the `ddof` it used, so a std is never reported without saying which formula produced it.

In [11]:
s1 = summarize(scores, ddof=1)
s0 = summarize(scores, ddof=0)
factor = (n / (n - 1)) ** 0.5

print(f"{'':<10}{'mean':>9}{'std':>10}{'ddof':>6}{'n':>4}")
print(f"{'ddof=1':<10}{s1['mean']:>9.4f}{s1['std']:>10.5f}{s1['ddof']:>6}{s1['n']:>4}   <- the team's choice")
print(f"{'ddof=0':<10}{s0['mean']:>9.4f}{s0['std']:>10.5f}{s0['ddof']:>6}{s0['n']:>4}")
print()
print(f"std(ddof=1) / std(ddof=0)       = {s1['std'] / s0['std']:.4f}")
print(f"sqrt(n / (n - 1)) for n = {n}      = {factor:.4f}")
print(f"so ddof=1 is {100 * (factor - 1):.1f}% larger than ddof=0 when n = {n}")

               mean       std  ddof   n
ddof=1       0.6280   0.01924     1   5   <- the team's choice
ddof=0       0.6280   0.01720     0   5

std(ddof=1) / std(ddof=0)       = 1.1180
sqrt(n / (n - 1)) for n = 5      = 1.1180
so ddof=1 is 11.8% larger than ddof=0 when n = 5


**`summarize_metrics`** does the same for a whole set of metrics at once. It takes one dictionary of metrics per seed and requires every seed to report the same metric names. This is what the run record uses.

`summarize` also refuses input that would give a meaningless std, for example a single value with `ddof=1`, or a value that is not a finite number.

In [12]:
per_seed_demo = [  # 🧪 synthetic: one dict per seed
    {"metric_a": 0.62, "metric_b": 0.50},
    {"metric_a": 0.65, "metric_b": 0.53},
    {"metric_a": 0.60, "metric_b": 0.49},
    {"metric_a": 0.64, "metric_b": 0.52},
    {"metric_a": 0.63, "metric_b": 0.51},
]
table = summarize_metrics(per_seed_demo, ddof=1)
print(f"{'metric':<10}{'mean':>9}{'std':>10}{'ddof':>6}{'n':>4}")
for name, s in table.items():
    print(f"{name:<10}{s['mean']:>9.4f}{s['std']:>10.5f}{s['ddof']:>6}{s['n']:>4}")

print()
for label, call in {
    "a single value, ddof=1": lambda: summarize([0.7], ddof=1),
    "a NaN value": lambda: summarize([0.7, float("nan")]),
    "seeds with different metrics": lambda: summarize_metrics([{"a": 0.5}, {"b": 0.6}]),
}.items():
    try:
        call()
        print(f"{label:<30}: accepted")
    except ValueError as exc:
        print(f"{label:<30}: REJECTED -> {exc}")

metric         mean       std  ddof   n
metric_a     0.6280   0.01924     1   5
metric_b     0.5100   0.01581     1   5

a single value, ddof=1        : REJECTED -> need more than 1 value(s) to compute std with ddof=1
a NaN value                   : REJECTED -> values must be finite numbers, got nan
seeds with different metrics  : REJECTED -> seeds report different metrics: ['a'] vs ['b']


> **What this module does not do.** It reports the mean and the std. It does not pick a winner. The tie-break rule between EgyBERT and MARBERTv2 is defined in the model selection report and applied by Hazem; it uses these numbers.

## 5. The experiment config

Hazem's example config has these fields: **model, learning rate, batch size, epochs, seed, dataset version, split**. `ExperimentConfig` keeps exactly those. Three details:

1. **The seed is not stored in the config.** One experiment is one config run once per seed, and the seeds come from `shared_config.json`. `for_seed(seed)` returns the full per-run config, with the seed added.
2. **Only the `train` split is accepted for training.** Anything else (`val`, `test`) is rejected, so nobody can train on the test set by mistake.
3. **Fields that do not apply are left as `None`.** The TF-IDF baseline has no learning rate, batch size or epochs. Anything model specific beyond those goes in `other_hyperparameters`.

In [13]:
EXAMPLE = dict(model="EgyBERT", learning_rate=2e-5, batch_size=16, epochs=3)  # 📋 Hazem's example
config = ExperimentConfig(
    dataset_version=shared["dataset_version"],  # ✅ from the shared config
    split="train",
    **EXAMPLE,
)

print("The experiment config (the same for all five seeds):")
print(json.dumps(config.to_dict(), indent=2))
print()
print(f"The config of one run, for seed {seeds[0]} (this is the shape in Hazem's example):")
print(json.dumps(config.for_seed(seeds[0]), indent=2))

The experiment config (the same for all five seeds):
{
  "model": "EgyBERT",
  "dataset_version": "v2",
  "split": "train",
  "learning_rate": 2e-05,
  "batch_size": 16,
  "epochs": 3,
  "other_hyperparameters": {}
}

The config of one run, for seed 42 (this is the shape in Hazem's example):
{
  "model": "EgyBERT",
  "dataset_version": "v2",
  "split": "train",
  "learning_rate": 2e-05,
  "batch_size": 16,
  "epochs": 3,
  "other_hyperparameters": {},
  "seed": 42
}


In [14]:
baseline = ExperimentConfig(
    model="tfidf-logreg",
    dataset_version=shared["dataset_version"],
    other_hyperparameters={"max_features": 20000, "C": 1.0},  # 🧪 illustrative values only
)
print("A non-neural model leaves the neural fields empty:")
print(json.dumps(baseline.to_dict(), indent=2))

A non-neural model leaves the neural fields empty:
{
  "model": "tfidf-logreg",
  "dataset_version": "v2",
  "split": "train",
  "learning_rate": null,
  "batch_size": null,
  "epochs": null,
  "other_hyperparameters": {
    "max_features": 20000,
    "C": 1.0
  }
}


**What the config rejects.** Each case below is a mistake that would otherwise slip into a run unnoticed.

In [15]:
def show(label, make):
    try:
        make()
        print(f"{label:<40}: accepted")
    except ExperimentConfigError as exc:
        print(f"{label:<40}: REJECTED -> {exc}")


ds = shared["dataset_version"]
show("split = 'test'", lambda: ExperimentConfig(model="m", dataset_version=ds, split="test"))
show("split = 'val'", lambda: ExperimentConfig(model="m", dataset_version=ds, split="val"))
show("learning_rate = 0", lambda: ExperimentConfig(model="m", dataset_version=ds, learning_rate=0))
show("batch_size = 1.5", lambda: ExperimentConfig(model="m", dataset_version=ds, batch_size=1.5))
show("seed inside the other hyperparameters",
     lambda: ExperimentConfig(model="m", dataset_version=ds, other_hyperparameters={"seed": 1}))
show("'seed' in a config file",
     lambda: ExperimentConfig.from_dict({"model": "m", "dataset_version": ds, "seed": 42}))
show("an unknown field (typo 'lr')",
     lambda: ExperimentConfig.from_dict({"model": "m", "dataset_version": ds, "lr": 1e-3}))

split = 'test'                          : REJECTED -> split must be one of ('train',), got 'test'
split = 'val'                           : REJECTED -> split must be one of ('train',), got 'val'
learning_rate = 0                       : REJECTED -> learning_rate must be a positive number or None, got 0
batch_size = 1.5                        : REJECTED -> batch_size must be a positive integer or None, got 1.5
seed inside the other hyperparameters   : REJECTED -> other_hyperparameters must not contain ['seed']; use the named fields (the seed comes from shared_config.json)
'seed' in a config file                 : REJECTED -> do not put 'seed' in the experiment config: seeds come from shared_config.json, use for_seed(seed)
an unknown field (typo 'lr')            : REJECTED -> unknown config fields: ['lr']


**`changed_fields(a, b)`** lists what differs between two configs. The run record uses it to enforce the rule **change one factor at a time**: if you compare a run against its reference and more than one thing changed, you cannot tell which one made the difference.

In [16]:
reference = config
print("same config            :", changed_fields(reference, replace(config)))
print("learning rate only     :", changed_fields(reference, replace(config, learning_rate=3e-5)))
print("model only             :", changed_fields(reference, replace(config, model="MARBERTv2")))
print("two things at once     :", changed_fields(reference, replace(config, learning_rate=3e-5, batch_size=8)))
print("a hyperparameter inside:", changed_fields(
    replace(config, other_hyperparameters={"warmup_ratio": 0.1}),
    replace(config, other_hyperparameters={"warmup_ratio": 0.06}),
))

same config            : []
learning rate only     : ['learning_rate']
model only             : ['model']
two things at once     : ['batch_size', 'learning_rate']
a hyperparameter inside: ['other_hyperparameters.warmup_ratio']


**Saving and loading.** A config file is plain JSON. `save_experiment_config` refuses to overwrite an existing file, so an old experiment's config can never change silently, and `load_experiment_config` can check the dataset version against the shared config at the same time.

In [17]:
folder = fresh_dir()
path = folder / "exp_config.json"
save_experiment_config(config, path)
print("saved:", path.name)
print("loaded config equals the saved one:", load_experiment_config(path, shared_config=shared) == config)

try:
    save_experiment_config(replace(config, epochs=5), path)
except FileExistsError:
    print("saving a different config to the same file -> REFUSED:", path.name, "already exists")

old = fresh_dir() / "old.json"
save_experiment_config(replace(config, dataset_version="v-old"), old)
try:
    load_experiment_config(old, shared_config=shared)
except ExperimentConfigError as exc:
    print("a config for another dataset version -> REJECTED:", exc)

saved: exp_config.json
loaded config equals the saved one: True
saving a different config to the same file -> REFUSED: exp_config.json already exists
a config for another dataset version -> REJECTED: dataset_version 'v-old' does not match shared_config 'v2'


## 6. Per-field metrics (`metrics.py`)

`evaluate_field(y_true, y_pred, classes)` computes the metrics specified in `research/recommendations.md` for one field:
- **macro-F1** (the headline metric), with macro-precision and macro-recall;
- **per-class** precision, recall and F1, with the true support count;
- the **confusion matrix** as raw counts (rows are true classes, columns are predicted classes, in the order of `classes`);
- **accuracy** for context only (micro-F1 is not reported separately because on single-label data it equals accuracy).

**Why `classes` is always passed in explicitly:** if a class happens to have 0 true rows in a split, inferring classes from the data would make it vanish silently. With an explicit list, `evaluate_field` keeps the class in the macro average (with F1 = 0) and lists it under `classes_without_support`. Conversely, a label that must not be scored (`unclear` in `customer_state`) is left out of `classes` and its rows are removed before scoring (see section 7).

The cell below works out a 6-row synthetic fixture 🧪 by hand and checks `evaluate_field` against both the hand calculation and `scikit-learn`.

In [18]:
import sklearn.metrics as sk_metrics

# 🧪 Synthetic 6-row fixture worked out by hand:
#   true:      a  a  a  b  b  c
#   predicted: a  a  b  b  c  c
# Confusion matrix (rows = true, columns = predicted):
#        a  b  c
#   a [  2, 1, 0 ]   precision=2/2=1.0, recall=2/3, f1=0.8
#   b [  0, 1, 1 ]   precision=1/2=0.5, recall=1/2, f1=0.5
#   c [  0, 0, 1 ]   precision=1/2=0.5, recall=1/1, f1=2/3
# macro-F1 = (0.8 + 0.5 + 2/3) / 3 = 0.655556, accuracy = 4/6 = 0.666667
toy_classes = ["a", "b", "c"]
y_true = ["a", "a", "a", "b", "b", "c"]
y_pred = ["a", "a", "b", "b", "c", "c"]

field_rep = evaluate_field(y_true, y_pred, toy_classes)
hand_macro_f1 = (0.8 + 0.5 + 2 / 3) / 3
sk_macro_f1 = float(sk_metrics.f1_score(y_true, y_pred, labels=toy_classes, average="macro", zero_division=0))

print("confusion matrix counts (rows=true, cols=predicted):")
for label, row in zip(toy_classes, field_rep["confusion_matrix"]["counts"]):
    print(f"  {label}: {row}")
print()
print("per-class metrics:")
for c, m in field_rep["per_class"].items():
    print(f"  {c}: precision={m['precision']:.4f}  recall={m['recall']:.4f}  f1={m['f1']:.4f}  support={m['support']}")
print()
print(f"macro-F1 by hand       : {hand_macro_f1:.6f}")
print(f"macro-F1 evaluate_field: {field_rep['macro']['f1']:.6f}")
print(f"macro-F1 scikit-learn  : {sk_macro_f1:.6f}")
print("all three agree        :", abs(field_rep["macro"]["f1"] - hand_macro_f1) < 1e-12
      and abs(field_rep["macro"]["f1"] - sk_macro_f1) < 1e-12)

confusion matrix counts (rows=true, cols=predicted):
  a: [2, 1, 0]
  b: [0, 1, 1]
  c: [0, 0, 1]

per-class metrics:
  a: precision=1.0000  recall=0.6667  f1=0.8000  support=3
  b: precision=0.5000  recall=0.5000  f1=0.5000  support=2
  c: precision=0.5000  recall=1.0000  f1=0.6667  support=1

macro-F1 by hand       : 0.655556
macro-F1 evaluate_field: 0.655556
macro-F1 scikit-learn  : 0.655556
all three agree        : True


**Combined score (`combine_macro_f1`) and flat seed metrics (`seed_metrics`).** `combine_macro_f1` computes the unweighted mean of the per-field macro-F1 values and always returns the individual field values next to it, so a weak field can never hide behind the average. `seed_metrics` formats the per-field macro-F1 and accuracy under the exact keys the run record expects.

In [19]:
two_fields = {
    "field_one": field_rep,
    "field_two": evaluate_field(y_true, y_true, toy_classes),  # 🧪 perfect on field_two
}
print("combine_macro_f1:")
print(json.dumps(combine_macro_f1(two_fields), indent=2))
print()
print("seed_metrics:")
print(json.dumps(seed_metrics(two_fields), indent=2))

combine_macro_f1:
{
  "label": "unweighted mean of the per-field macro-F1 values",
  "mean": 0.8277777777777777,
  "values": {
    "field_one": 0.6555555555555556,
    "field_two": 1.0
  }
}

seed_metrics:
{
  "field_one_macro_f1": 0.6555555555555556,
  "field_one_accuracy": 0.6666666666666666,
  "field_two_macro_f1": 1.0,
  "field_two_accuracy": 1.0
}


## 7. The evaluation harness and mock predictions (`harness.py`, `mock_predictions.py`)

### 7.1 Reading the approved class lists from `final/class_weights.json`

Per project rules, the class lists of the four fields are **read from `final/class_weights.json`, never hard-coded**. Hazem recomputed `class_weights.json` on the `train` split only (3,575 rows):
- `complaint_category` has 11 classes;
- `complaint_urgency` has 3 classes (`high`, `low`, `medium`);
- `customer_state` has **4 classes** (`calm`, `frustrated`, `mad`, `satisfied`). `unclear` has 0 rows in `train`, so no model trains on it and it has no weight;
- `churn_risk` has 2 classes (`at_risk`, `not_at_risk`).

In [20]:
WEIGHTS_CANDIDATES = [
    REPO_ROOT / "final" / "class_weights.json",
    REPO_ROOT.parent / "svr-data" / "final" / "class_weights.json",
    REPO_ROOT.parent / "smart-voicemail-router" / "final" / "class_weights.json",
]
weights_path = next((p for p in WEIGHTS_CANDIDATES if p.is_file()), None)
if weights_path is None:
    raise FileNotFoundError("final/class_weights.json not found in repo, svr-data, or smart-voicemail-router.")

class_weights_data = json.loads(weights_path.read_text(encoding="utf-8"))
CLASSES_BY_FIELD = {
    field: list(weights.keys())
    for field, weights in class_weights_data["weights"].items()
}

print(f"loaded class lists from {weights_path.name} (train rows = {class_weights_data['rows']})  ✅")
print()
for field, classes in CLASSES_BY_FIELD.items():
    print(f"  {field:<20} ({len(classes):>2} classes): {classes}")
print()
print("notes in class_weights.json:")
for k, note in class_weights_data.get("notes", {}).items():
    print(f"  - {k}: {note}")

loaded class lists from class_weights.json (train rows = 3575)  ✅

  complaint_category   (11 classes): ['app_and_account', 'balance_and_data', 'billing_and_charges', 'branch_and_staff', 'cancellation_request', 'network_and_connectivity', 'no_reply_or_long_wait', 'offers_and_packages', 'other', 'sim_services', 'thanks_or_praise']
  complaint_urgency    ( 3 classes): ['high', 'low', 'medium']
  customer_state       ( 4 classes): ['calm', 'frustrated', 'mad', 'satisfied']
  churn_risk           ( 2 classes): ['at_risk', 'not_at_risk']

notes in class_weights.json:
  - customer_state unclear: 0 rows in train (1 row in val, 0 in test). No weight: the model never trains on unclear. Review the val row before the retrain on audited labels.
  - evaluation: Macro-F1 for customer_state is computed over the 4 classes present in train (calm, frustrated, mad, satisfied). Unclear is excluded from the denominator because it has 0 train rows.


### 7.2 Scoring predictions and recording the `unclear` exception

`evaluate_predictions(split, truth_rows, pred_rows, classes_by_field, exclusions_by_field=...)` matches rows by `audio_file` **inside one split** (because `audio_file` names repeat across splits for different recordings).

By Hazem's decision:
- `customer_state` is scored on the **4 classes** (`calm`, `frustrated`, `mad`, `satisfied`).
- Any row whose true `customer_state` is `unclear` (1 row in `val`, 0 in `test`) is **excluded from the `customer_state` score only**, and **kept in the other three fields**.
- Even on an excluded row, the model's prediction for `customer_state` must still be one of the 4 classes.
- The harness records every exclusion under `report["exceptions"]` with the field name, the excluded label, `rows_excluded`, `n_scored`, `n_total_rows` and the reason, and always includes `label_status` stating that the labels come from a model and are being audited.

Below we build a 7-row synthetic fixture 🧪 (including 1 row with `customer_state="unclear"`), generate deterministic mock predictions with `make_mock_predictions(..., wrong_every=3)`, and run the harness.

In [21]:
UNCLEAR_REASON = (
    "unclear has 0 rows in train (1 in val, 0 in test), so the model never trains on it; "
    "excluded from customer_state macro-F1 only (Hazem's decision)."
)
EXCLUSIONS = {
    "customer_state": {
        "labels": ["unclear"],
        "reason": UNCLEAR_REASON,
    }
}

# 🧪 7 synthetic validation rows; row 7 has true customer_state = "unclear".
cats = CLASSES_BY_FIELD["complaint_category"]
urgs = CLASSES_BY_FIELD["complaint_urgency"]
states = CLASSES_BY_FIELD["customer_state"]
churns = CLASSES_BY_FIELD["churn_risk"]

truth_fixture = [
    {
        "audio_file": f"sample_{i:04d}.wav",
        "complaint_category": cats[i % len(cats)],
        "complaint_urgency": urgs[i % len(urgs)],
        "customer_state": "unclear" if i == 7 else states[i % len(states)],
        "churn_risk": churns[i % len(churns)],
    }
    for i in range(1, 8)
]

mock_preds = make_mock_predictions(
    truth_fixture,
    CLASSES_BY_FIELD,
    wrong_every=3,
    exclusions_by_field=EXCLUSIONS,
)

harness_report = evaluate_predictions(
    "val",
    truth_fixture,
    mock_preds,
    CLASSES_BY_FIELD,
    exclusions_by_field=EXCLUSIONS,
)

print("split        :", harness_report["split"])
print("total rows   :", harness_report["n"])
print("label_status :", harness_report["label_status"])
print()
print(f"{'field':<22}{'n_scored':>10}{'n_excluded':>12}{'macro_f1':>12}{'accuracy':>12}")
for field, r in harness_report["fields"].items():
    print(f"{field:<22}{r['n']:>10}{r['n_excluded']:>12}{r['macro']['f1']:>12.4f}{r['accuracy']:>12.4f}")
print()
print("combined_macro_f1:")
print(json.dumps(harness_report["combined_macro_f1"], indent=2))
print()
print("exceptions recorded by the harness:")
print(json.dumps(harness_report["exceptions"], indent=2))

split        : val
total rows   : 7
label_status : The labels come from a model and are still being audited by the team. They are not human-verified until the audit approves them.

field                   n_scored  n_excluded    macro_f1    accuracy
complaint_category             7           0      0.3939      0.7143
complaint_urgency              7           0      0.5833      0.7143
customer_state                 6           1      0.5833      0.6667
churn_risk                     7           0      0.7083      0.7143

combined_macro_f1:
{
  "label": "unweighted mean of the per-field macro-F1 values",
  "mean": 0.5672348484848484,
  "values": {
    "complaint_category": 0.3939393939393939,
    "complaint_urgency": 0.5833333333333334,
    "customer_state": 0.5833333333333333,
    "churn_risk": 0.7083333333333333
  }
}

exceptions recorded by the harness:
[
  {
    "field": "customer_state",
    "excluded_labels": [
      "unclear"
    ],
    "rows_excluded": 1,
    "n_scored": 6,
    

**Loading true labels without reading text (`load_truth_table`).** When scoring a real split, `load_truth_table(path, split, fields)` passes `usecols=["audio_file", "split", *fields]` to `pandas.read_csv`, so the transcript columns (`text`, `text_norm`) are never loaded into memory.

In [22]:
import csv

csv_path = fresh_dir() / "clean_training_val.csv"
with csv_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["audio_file", "split", "text_norm", *CLASSES_BY_FIELD])
    writer.writeheader()
    for row in truth_fixture[:2]:
        writer.writerow({**row, "split": "val", "text_norm": "SYNTHETIC_TRANSCRIPT_NEVER_LOADED"})

loaded_truth = load_truth_table(csv_path, "val", list(CLASSES_BY_FIELD))
print("loaded rows:", len(loaded_truth))
print("first row keys:", list(loaded_truth[0].keys()))
print("transcript present in loaded data:", "SYNTHETIC_TRANSCRIPT" in str(loaded_truth))

loaded rows: 2
first row keys: ['audio_file', 'complaint_category', 'complaint_urgency', 'customer_state', 'churn_risk']
transcript present in loaded data: False


**What the harness rejects.**

In [23]:
def try_harness(label, **overrides):
    args = dict(
        split="val",
        truth_rows=truth_fixture,
        pred_rows=mock_preds,
        classes_by_field=CLASSES_BY_FIELD,
        exclusions_by_field=EXCLUSIONS,
    )
    args.update(overrides)
    try:
        evaluate_predictions(**args)
        print(f"{label:<44}: accepted")
    except HarnessError as exc:
        print(f"{label:<44}: REJECTED -> {exc}")


try_harness("valid mock predictions")
try_harness("scoring the train split", split="train")
try_harness("missing one prediction row", pred_rows=mock_preds[:-1])
try_harness("duplicate audio_file in predictions", pred_rows=[*mock_preds, mock_preds[0]])

bad_label_preds = [dict(r) for r in mock_preds]
bad_label_preds[0] = {**bad_label_preds[0], "complaint_urgency": "urgent"}
try_harness("predicted label outside the class list", pred_rows=bad_label_preds)

unclear_pred_on_excluded = [dict(r) for r in mock_preds]
unclear_pred_on_excluded[-1] = {**unclear_pred_on_excluded[-1], "customer_state": "unclear"}
try_harness("predicting 'unclear' on the excluded row", pred_rows=unclear_pred_on_excluded)

valid mock predictions                      : accepted
scoring the train split                     : REJECTED -> split must be one of ('val', 'test'), got 'train'
missing one prediction row                  : REJECTED -> 1 true rows have no prediction, for example ['sample_0007.wav']
duplicate audio_file in predictions         : REJECTED -> pred_rows has the audio_file 'sample_0001.wav' more than once
predicted label outside the class list      : REJECTED -> field complaint_urgency: the prediction for 'sample_0001.wav' is 'urgent', which is not in the class list ['high', 'low', 'medium']
predicting 'unclear' on the excluded row    : REJECTED -> field customer_state: the prediction for 'sample_0007.wav' is 'unclear', which is not in the class list ['calm', 'frustrated', 'mad', 'satisfied']


## 8. The predictions file (`predictions_file.py`)

Every evaluated seed writes one JSON file: `runs/exp_<run_id>/predictions_<split>_seed<seed>.json`. The format was agreed with Hazem:
- **Header:** `schema_version`, `run_id`, `model`, `split` (`val` or `test` only), `seed`, `dataset_version`.
- **Rows:** one entry per `audio_file` in that split, with `labels` and `probabilities`.
- **Forced probabilities for all four fields:** every row must hold the probability of every class of every field, between `0` and `1` and summing to `1` (within `1e-4`). Nothing is optional, because threshold tuning and average precision require the probabilities.
- **Labels are derived from probabilities:** `write_predictions_file` takes only `audio_file` and `probabilities` for each row and computes `labels[field]` as the class with the highest probability (ties go to the first class in the field's class list). `read_predictions_file` checks that `labels` and `probabilities` agree.
- **No text allowed:** any extra key (such as `text` or `text_norm`) is rejected by both the writer and the reader.

In [24]:
def make_synthetic_probabilities(truth_rows, classes_by_field, seed, confidence=0.75):
    """🧪 Build synthetic probability dicts per row so we can test the predictions file."""
    rng = np.random.default_rng(seed)
    out = []
    for row in truth_rows:
        probs = {}
        for field, classes in classes_by_field.items():
            # Pick a target class from the allowed classes (if true label is 'unclear', pick classes[0]).
            true_label = row[field]
            target = true_label if true_label in classes else classes[0]
            # Small deterministic noise per seed so seeds differ slightly.
            if rng.random() < 0.25:
                target = classes[(classes.index(target) + 1) % len(classes)]
            rest = (1.0 - confidence) / (len(classes) - 1)
            probs[field] = {c: (confidence if c == target else rest) for c in classes}
        out.append({"audio_file": row["audio_file"], "probabilities": probs})
    return out


pred_dir = fresh_dir()
pred_file = predictions_path(pred_dir, "001", "val", seeds[0])
prob_rows = make_synthetic_probabilities(truth_fixture, CLASSES_BY_FIELD, seed=seeds[0])

write_predictions_file(
    path=pred_file,
    run_id="001",
    model="EgyBERT",
    split="val",
    seed=seeds[0],
    dataset_version=shared["dataset_version"],
    rows=prob_rows,
    classes_by_field=CLASSES_BY_FIELD,
)

loaded_pred = read_predictions_file(pred_file, CLASSES_BY_FIELD, expected_split="val")
print("saved to           :", pred_file.relative_to(pred_dir).as_posix())
print("header meta        :", loaded_pred["meta"])
print("first harness_row  :", loaded_pred["harness_rows"][0])
print("churn probs (row 0):", loaded_pred["probabilities"]["sample_0001.wav"]["churn_risk"])

# Feed loaded_pred["harness_rows"] straight into evaluate_predictions:
rep_from_file = evaluate_predictions(
    "val", truth_fixture, loaded_pred["harness_rows"], CLASSES_BY_FIELD, exclusions_by_field=EXCLUSIONS
)
print("scored by harness  : combined mean macro-F1 =", round(rep_from_file["combined_macro_f1"]["mean"], 4))

saved to           : exp_001/predictions_val_seed42.json
header meta        : {'schema_version': 1, 'run_id': '001', 'model': 'EgyBERT', 'split': 'val', 'seed': 42, 'dataset_version': 'v2', 'n_rows': 7}
first harness_row  : {'audio_file': 'sample_0001.wav', 'complaint_category': 'balance_and_data', 'complaint_urgency': 'low', 'customer_state': 'frustrated', 'churn_risk': 'not_at_risk'}
churn probs (row 0): {'at_risk': 0.25, 'not_at_risk': 0.75}
scored by harness  : combined mean macro-F1 = 0.7081


**What the predictions file writer and reader reject.**

In [25]:
def try_write_pred(label, edit_rows=None, **overrides):
    rows_copy = json.loads(json.dumps(prob_rows))
    if edit_rows is not None:
        edit_rows(rows_copy)
    args = dict(
        path=fresh_dir() / "pred.json",
        run_id="001",
        model="EgyBERT",
        split="val",
        seed=seeds[0],
        dataset_version=shared["dataset_version"],
        rows=rows_copy,
        classes_by_field=CLASSES_BY_FIELD,
    )
    args.update(overrides)
    try:
        write_predictions_file(**args)
        print(f"{label:<45}: accepted")
    except (PredictionsFileError, FileExistsError) as exc:
        print(f"{label:<45}: REJECTED -> {exc}")


try_write_pred("valid predictions file")
try_write_pred(
    "missing probabilities for churn_risk",
    lambda rs: rs[0]["probabilities"].pop("churn_risk"),
)
try_write_pred(
    "probabilities do not add up to 1",
    lambda rs: rs[0]["probabilities"]["churn_risk"].update(at_risk=0.5, not_at_risk=0.2),
)
try_write_pred(
    "'unclear' included in customer_state probs",
    lambda rs: rs[0]["probabilities"]["customer_state"].update(unclear=0.0),
)
try_write_pred(
    "extra 'text' key inside a row",
    lambda rs: rs[0].update(text="RAW_TRANSCRIPT"),
)
try_write_pred("split = 'train'", split="train")

try:
    read_predictions_file(pred_file, CLASSES_BY_FIELD, expected_split="test")
except PredictionsFileError as exc:
    print(f"{'reading a val file when test was expected':<45}: REJECTED -> {exc}")

valid predictions file                       : accepted
missing probabilities for churn_risk         : REJECTED -> rows[0] (sample_0001.wav): probabilities are forced for every field, exactly ['churn_risk', 'complaint_category', 'complaint_urgency', 'customer_state']
probabilities do not add up to 1             : REJECTED -> rows[0] (sample_0001.wav): probabilities of churn_risk add up to 0.700000, not 1
'unclear' included in customer_state probs   : REJECTED -> rows[0] (sample_0001.wav): probabilities of customer_state must name exactly the classes ['calm', 'frustrated', 'mad', 'satisfied']
extra 'text' key inside a row                : REJECTED -> rows[0] must have exactly 'audio_file' and 'probabilities'
split = 'train'                              : REJECTED -> split must be one of ('val', 'test'), got 'train'
reading a val file when test was expected    : REJECTED -> this file is for split 'val' but 'test' was expected


## 9. The run record (`run_record.py`, schema version 2)

Every experiment has one run record, a single JSON file (`schema_version = 2`). Hazem's version has six parts, plus the dataset metadata, environment, and **`exceptions`**:

| Part | Meaning |
|---|---|
| **run ID** | The experiment's name, for example `001`. |
| **hypothesis** | What you expect, in one sentence. |
| **changed factor** | The one thing that differs from the reference run (`None` for a reference run). |
| **held-constant conditions** | Everything else, which stayed the same. |
| **exceptions** | Every rule that excluded rows from a field's score (taken from the harness report, or `[]` when nothing was excluded). Required so an exclusion is never forgotten. |
| **measures** | The macro-F1 for each of the four fields, per seed, with the mean and sample std (`ddof=1`). |
| **artifacts** | Relative paths to the per-seed prediction files and checkpoints. |

**Computed, not typed.** `held_constant` is derived from the config. The mean and std are computed from the per-seed results. The check that exactly one factor changed compares the config with the reference config. The `exceptions` row counts (`rows_excluded + n_scored == n_total_rows`) are checked against the evaluated split's row count in `shared_config.json`.

### 9.1 A complete example

> **🧪 Every metric value in this section is synthetic.** The numbers are deliberately made up and unrelated to any real model. The hypothesis below is Hazem's example text; the made-up numbers prove nothing about it.

In [26]:
SYNTHETIC = {  # 🧪 one value per seed, in the order of the shared config's seeds
    "complaint_category_macro_f1": [0.60, 0.62, 0.58, 0.61, 0.59],
    "complaint_urgency_macro_f1": [0.50, 0.52, 0.48, 0.51, 0.49],
    "customer_state_macro_f1": [0.40, 0.42, 0.38, 0.41, 0.39],
    "churn_risk_macro_f1": [0.30, 0.34, 0.26, 0.32, 0.28],
}
assert len(seeds) == 5, "this example has five values per metric; adjust it if the seed list changes"

per_seed = [
    make_seed_result("001", config, "val", seed,
                     {name: values[i] for name, values in SYNTHETIC.items()})
    for i, seed in enumerate(seeds)
]

# In validation (1,003 rows in shared_config.json), 1 row has true customer_state = 'unclear'.
VAL_EXCEPTIONS = [{
    "field": "customer_state",
    "excluded_labels": ["unclear"],
    "rows_excluded": 1,
    "n_scored": shared["val_rows"] - 1,
    "n_total_rows": shared["val_rows"],
    "reason": UNCLEAR_REASON,
}]

print("One seed's result (the shape of make_result_record in the training skeleton):")
print(json.dumps(per_seed[0], indent=2))

One seed's result (the shape of make_result_record in the training skeleton):
{
  "run_id": "001",
  "model": "EgyBERT",
  "split": "val",
  "seed": 42,
  "metrics": {
    "complaint_category_macro_f1": 0.6,
    "complaint_urgency_macro_f1": 0.5,
    "customer_state_macro_f1": 0.4,
    "churn_risk_macro_f1": 0.3
  }
}


In [27]:
marbert = replace(config, model="MARBERTv2")  # the reference run: the same config with the other model

record = build_run_record(
    run_id="001",
    hypothesis="EgyBERT beats MARBERTv2 on complaint_category",
    config=config,
    shared_config=shared,
    per_seed=per_seed,
    evaluated_split="val",
    artifacts={
        "checkpoints": [f"runs/exp_001/checkpoints/seed_{s}" for s in seeds],
        "predictions": [f"runs/exp_001/predictions_val_seed{s}.json" for s in seeds],
    },
    device="cpu",
    exceptions=VAL_EXCEPTIONS,
    reference_config=marbert,
    changed_factor="model",
    reference_run_id="002",  # the run that produced the MARBERTv2 numbers
)
print("record built (schema_version =", record["schema_version"], "). Top-level parts:")
for key in record:
    print("  -", key)

record built (schema_version = 2 ). Top-level parts:
  - schema_version
  - run_id
  - hypothesis
  - changed_factor
  - held_constant
  - reference_run_id
  - config
  - dataset
  - evaluated_split
  - exceptions
  - per_seed
  - aggregate
  - artifacts
  - environment
  - created_utc


In [28]:
print("run_id          :", record["run_id"])
print("hypothesis      :", record["hypothesis"])
print("changed_factor  :", record["changed_factor"])
print("held_constant   :", record["held_constant"])
print("evaluated_split :", record["evaluated_split"], "(the split the numbers come from)")
print("exceptions      :", record["exceptions"])
print("dataset         :", record["dataset"], "  <- ✅ from the shared config")
print("environment     :", record["environment"])
print()
print("`held_constant` was derived: the config minus the changed factor.")
print("Note that `split` in the config is the TRAINING split; `evaluated_split` is where the numbers come from.")

run_id          : 001
hypothesis      : EgyBERT beats MARBERTv2 on complaint_category
changed_factor  : model
held_constant   : {'dataset_version': 'v2', 'split': 'train', 'learning_rate': 2e-05, 'batch_size': 16, 'epochs': 3}
evaluated_split : val (the split the numbers come from)
exceptions      : [{'field': 'customer_state', 'excluded_labels': ['unclear'], 'rows_excluded': 1, 'n_scored': 1002, 'n_total_rows': 1003, 'reason': "unclear has 0 rows in train (1 in val, 0 in test), so the model never trains on it; excluded from customer_state macro-F1 only (Hazem's decision)."}]
dataset         : {'version': 'v2', 'rows': {'train': 3575, 'val': 1003, 'test': 518}}   <- ✅ from the shared config
environment     : {'python': '3.11.9', 'platform': 'Windows-10-10.0.26200-SP0', 'device': 'cpu', 'packages': {'numpy': '1.26.4', 'pandas': '2.1.4', 'scikit-learn': '1.9.1'}}

`held_constant` was derived: the config minus the changed factor.
Note that `split` in the config is the TRAINING split; `eva

In [29]:
print("aggregate (computed from the five seeds, ddof recorded):")
print(f"  {'metric':<30}{'mean':>8}{'std':>9}{'ddof':>6}{'n':>4}")
for name, s in record["aggregate"].items():
    print(f"  {name:<30}{s['mean']:>8.4f}{s['std']:>9.5f}{s['ddof']:>6}{s['n']:>4}")

print()
print("per seed:")
names = list(SYNTHETIC)
print("  " + f"{'seed':<7}" + "".join(f"{n.replace('_macro_f1', ''):>22}" for n in names))
for r in record["per_seed"]:
    print("  " + f"{r['seed']:<7}" + "".join(f"{r['metrics'][n]:>22.2f}" for n in names))

aggregate (computed from the five seeds, ddof recorded):
  metric                            mean      std  ddof   n
  churn_risk_macro_f1             0.3000  0.03162     1   5
  complaint_category_macro_f1     0.6000  0.01581     1   5
  complaint_urgency_macro_f1      0.5000  0.01581     1   5
  customer_state_macro_f1         0.4000  0.01581     1   5

per seed:
  seed       complaint_category     complaint_urgency        customer_state            churn_risk
  42                       0.60                  0.50                  0.40                  0.30
  1337                     0.62                  0.52                  0.42                  0.34
  2023                     0.58                  0.48                  0.38                  0.26
  7                        0.61                  0.51                  0.41                  0.32
  99                       0.59                  0.49                  0.39                  0.28


### 9.2 What the run record rejects

Each case below changes one thing in the correct example and shows the error. These are the mistakes the record is designed to catch before they reach a report.

In [30]:
def build_with(**changes):
    args = dict(
        run_id="001",
        hypothesis="EgyBERT beats MARBERTv2 on complaint_category",
        config=config,
        shared_config=shared,
        per_seed=per_seed,
        evaluated_split="val",
        artifacts={"predictions": [f"runs/exp_001/predictions_val_seed{s}.json" for s in seeds]},
        device="cpu",
        exceptions=VAL_EXCEPTIONS,
        reference_config=marbert,
        changed_factor="model",
    )
    args.update(changes)
    return build_run_record(**args)


def refuse(label, **changes):
    try:
        build_with(**changes)
        print(f"{label:<38}: accepted")
    except (RunRecordError, ExperimentConfigError) as exc:
        print(f"{label:<38}: REJECTED -> {exc}")


refuse("the correct example")
refuse("only four seeds", per_seed=per_seed[:-1])
refuse("seeds in a different order", per_seed=[per_seed[1], per_seed[0], *per_seed[2:]])
refuse("two factors changed", reference_config=replace(marbert, learning_rate=3e-5))
refuse("nothing changed", reference_config=config)
refuse("changed_factor names the wrong thing", changed_factor="learning_rate")
refuse("a factor but no reference config", reference_config=None)

missing_metric = [
    {**r, "metrics": {k: v for k, v in r["metrics"].items() if k != "churn_risk_macro_f1"}}
    for r in per_seed
]
refuse("a required metric is missing", per_seed=missing_metric)

too_high = [
    {**r, "metrics": {**r["metrics"], "customer_state_macro_f1": 1.2}} if i == 0 else r
    for i, r in enumerate(per_seed)
]
refuse("a macro-F1 above 1", per_seed=too_high)
refuse("exception counts do not add up", exceptions=[{**VAL_EXCEPTIONS[0], "n_scored": 900}])
refuse("exception total != split rows", exceptions=[{**VAL_EXCEPTIONS[0], "n_scored": 499, "n_total_rows": 500}])
refuse("no device written", device="")
refuse("an absolute artifact path", artifacts={"predictions": ["C:/Users/someone/pred.json"]})
refuse("an artifact name with 'transcript'", artifacts={"predictions": ["runs/exp_001/val_transcript_check.json"]})

the correct example                   : accepted
only four seeds                       : REJECTED -> every experiment must run all seeds in order: expected [42, 1337, 2023, 7, 99], got [42, 1337, 2023, 7]
seeds in a different order            : REJECTED -> every experiment must run all seeds in order: expected [42, 1337, 2023, 7, 99], got [1337, 42, 2023, 7, 99]
two factors changed                   : REJECTED -> more than one factor changed: ['learning_rate', 'model']
nothing changed                       : REJECTED -> nothing changed compared to the reference config
changed_factor names the wrong thing  : REJECTED -> changed_factor is 'learning_rate' but the configs differ in 'model'
a factor but no reference config      : REJECTED -> changed_factor needs a reference_config to check it against
a required metric is missing          : REJECTED -> seed 42: missing metrics ['churn_risk_macro_f1']
a macro-F1 above 1                    : REJECTED -> seed 42: customer_state_macro_f1 must be

### 9.3 Saving and loading

A run record is saved as `runs/exp_<run_id>/run_record.json` and **never overwritten**. The cell below saves into a temporary folder, shows the layout and the start of the file, and loads it back.

In [31]:
runs_dir = fresh_dir()
saved = save_run_record(record, runs_dir)
print("saved as:", saved.relative_to(runs_dir).as_posix())
print("loads back identical:", load_run_record(saved) == record)

try:
    save_run_record(record, runs_dir)
except FileExistsError:
    print("saving the same run again -> REFUSED (never overwritten)")

print()
print("first 38 lines of the file:")
print("\n".join(saved.read_text(encoding="utf-8").splitlines()[:38]))

saved as: exp_001/run_record.json
loads back identical: True
saving the same run again -> REFUSED (never overwritten)

first 38 lines of the file:
{
  "schema_version": 2,
  "run_id": "001",
  "hypothesis": "EgyBERT beats MARBERTv2 on complaint_category",
  "changed_factor": "model",
  "held_constant": {
    "dataset_version": "v2",
    "split": "train",
    "learning_rate": 2e-05,
    "batch_size": 16,
    "epochs": 3
  },
  "reference_run_id": "002",
  "config": {
    "model": "EgyBERT",
    "dataset_version": "v2",
    "split": "train",
    "learning_rate": 2e-05,
    "batch_size": 16,
    "epochs": 3,
    "other_hyperparameters": {}
  },
  "dataset": {
    "version": "v2",
    "rows": {
      "train": 3575,
      "val": 1003,
      "test": 518
    }
  },
  "evaluated_split": "val",
  "exceptions": [
    {
      "field": "customer_state",
      "excluded_labels": [
        "unclear"
      ],
      "rows_excluded": 1,


**What goes into Git, and what does not.** The team `.gitignore` decides:

| File | In Git? | Why |
|---|---|---|
| `run_record.json`, `predictions_<split>_seed<seed>.json` | ✅ yes | Hazem reviews them, and Ibthal uses the final predictions. |
| checkpoints (`checkpoints/`, `*.pt`, `*.safetensors`, `*.pkl`, ...) | ❌ no | Large. The record stores only their **relative path**. |
| `.csv` files | ❌ no | Ignored. Save predictions as JSON. |
| `*transcript*.json`, `*transcript*.txt`, `*transcript*.csv` | ❌ no | Ignored, and may hold raw text. The record rejects any artifact name containing `transcript`. |

Keep raw voicemail text out of the files you commit. Predictions refer to a sample by `split` and `audio_file` (sample names repeat between splits, so use both), never by copying its text.

## 10. A template for your own experiment

This is the full end-to-end flow on a synthetic validation table 🧪 (sized to `shared["val_rows"]` so the run record's split-size check passes, with 1 `unclear` row in `customer_state`).

**To run a real experiment, replace the body of `predict_probabilities(seed)`** with your training and validation code. Everything around it (writing the per-seed prediction file, reading and scoring it through the harness with the `unclear` exclusion, and building and saving the run record) stays the same.

The example runs a reference config (`2e-5`) and the two learning-rate sweep values from the roles guide (`1e-5` and `3e-5`, one changed factor each). All numbers are **🧪 synthetic**.

In [32]:
# 🧪 Synthetic validation truth table with shared["val_rows"] rows (1,003 rows, 1 'unclear' in customer_state).
full_val_truth = [
    {
        "audio_file": f"val_{i:04d}.wav",
        "complaint_category": cats[i % len(cats)],
        "complaint_urgency": urgs[i % len(urgs)],
        "customer_state": "unclear" if i == 1 else states[i % len(states)],
        "churn_risk": churns[i % len(churns)],
    }
    for i in range(1, shared["val_rows"] + 1)
]

sweep_dir = fresh_dir()
RUN_SPECS = [
    ("001", config, None, None, None, "Reference EgyBERT run at learning rate 2e-5"),
    ("003", replace(config, learning_rate=1e-5), config, "learning_rate", "001",
     "learning rate 1e-5 changes validation macro-F1 compared with 2e-5"),
    ("004", replace(config, learning_rate=3e-5), config, "learning_rate", "001",
     "learning rate 3e-5 changes validation macro-F1 compared with 2e-5"),
]

for run_id, run_config, ref_cfg, factor, ref_id, hypothesis in RUN_SPECS:
    reports_for_run = []
    pred_paths_rel = []

    def run_one_seed(seed):
        # 1. Produce per-class probabilities on validation (replace with real model training + inference).
        prob_rows_seed = make_synthetic_probabilities(
            full_val_truth, CLASSES_BY_FIELD, seed=seed, confidence=0.72
        )
        # 2. Write the predictions file for this seed.
        p_path = predictions_path(sweep_dir, run_id, "val", seed)
        write_predictions_file(
            path=p_path,
            run_id=run_id,
            model=run_config.model,
            split="val",
            seed=seed,
            dataset_version=shared["dataset_version"],
            rows=prob_rows_seed,
            classes_by_field=CLASSES_BY_FIELD,
        )
        pred_paths_rel.append(f"runs/exp_{run_id}/{p_path.name}")
        # 3. Read back and score with the evaluation harness.
        loaded = read_predictions_file(p_path, CLASSES_BY_FIELD, expected_split="val")
        rep = evaluate_predictions(
            "val", full_val_truth, loaded["harness_rows"], CLASSES_BY_FIELD, exclusions_by_field=EXCLUSIONS
        )
        reports_for_run.append(rep)
        return seed_metrics(rep["fields"])

    # 4. Run across all five seeds in order.
    results = run_over_seeds(run_one_seed, shared["experiment_seeds"])
    seed_results = [
        make_seed_result(run_id, run_config, "val", r["seed"], r["result"]) for r in results
    ]
    # 5. Build and save the run record (with the harness exceptions).
    rec = build_run_record(
        run_id=run_id,
        hypothesis=hypothesis,
        config=run_config,
        shared_config=shared,
        per_seed=seed_results,
        evaluated_split="val",
        artifacts={"predictions": pred_paths_rel},
        device="cpu",
        exceptions=reports_for_run[0]["exceptions"],
        reference_config=ref_cfg,
        changed_factor=factor,
        reference_run_id=ref_id,
    )
    save_run_record(rec, sweep_dir)
    print(f"run {run_id} saved (changed_factor={rec['changed_factor']!r}, lr={run_config.learning_rate:g}, "
          f"predictions files={len(pred_paths_rel)})")

run 001 saved (changed_factor=None, lr=2e-05, predictions files=5)
run 003 saved (changed_factor='learning_rate', lr=1e-05, predictions files=5)
run 004 saved (changed_factor='learning_rate', lr=3e-05, predictions files=5)


In [33]:
print("🧪 synthetic numbers, shown only to illustrate how runs are read side by side")
print(f"{'run':<5}{'changed factor':<17}{'lr':>9}   complaint_category_macro_f1 (mean ± std, ddof=1, n=5)   customer_state n_scored")
for run_dir in sorted(sweep_dir.glob("exp_*")):
    rec = load_run_record(run_dir / "run_record.json")
    agg = rec["aggregate"]["complaint_category_macro_f1"]
    exc = rec["exceptions"][0]
    factor_str = str(rec["changed_factor"])
    print(f"{rec['run_id']:<5}{factor_str:<17}{rec['config']['learning_rate']:>9g}   "
          f"{agg['mean']:.4f} ± {agg['std']:.4f}{'':<38}{exc['n_scored']}/{exc['n_total_rows']}")

🧪 synthetic numbers, shown only to illustrate how runs are read side by side
run  changed factor          lr   complaint_category_macro_f1 (mean ± std, ddof=1, n=5)   customer_state n_scored
001  None                 2e-05   0.7555 ± 0.0114                                      1002/1003
003  learning_rate        1e-05   0.7555 ± 0.0114                                      1002/1003
004  learning_rate        3e-05   0.7555 ± 0.0114                                      1002/1003


## 11. Rules, and what is still to come

**Rules from the roles guide and Hazem's decisions that these scripts enforce**

| Rule | Where it is enforced |
|---|---|
| Every experiment runs on all five seeds, in order. | `build_run_record` rejects a run without the exact seeds of `shared_config.json`. |
| Standard deviation across seeds is sample std (`ddof=1`). | `summarize` and `summarize_metrics` default to `ddof=1` and record `ddof` and `n`. |
| Change one factor at a time. | `changed_fields` plus `build_run_record`. |
| Never train on the test set. | `ExperimentConfig` accepts only `split="train"`. |
| Never score the train split as an evaluation result. | `evaluate_predictions` and `predictions_file` accept only `val` and `test`. |
| Class lists are explicit and read from `final/class_weights.json`. | `evaluate_field`, `evaluate_predictions` and `predictions_file` require explicit class lists. |
| `customer_state` is scored on the 4 classes; `unclear` is excluded from `customer_state` only and recorded as an exception. | `evaluate_predictions` (`exclusions_by_field`) and `build_run_record` (required `exceptions`, `schema_version = 2`). |
| Every evaluation report states that labels come from a model and are under audit. | `evaluate_predictions` includes `label_status` on every report. |
| Prediction files carry forced per-class probabilities for all four fields, and labels match the argmax. | `write_predictions_file` and `read_predictions_file`. |
| No data, audio or raw text in Git. | `load_truth_table` never loads transcript columns; `predictions_file` rejects extra keys such as `text`; `build_run_record` rejects absolute paths and `transcript` artifact names. |

**What these scripts do not do (yet)**
- They do not train a real model yet. The **TF-IDF + logistic regression baseline** (one model per field, trained on `train` and evaluated on `val` through these exact modules) is the next step.
- They do not yet compute the **churn-specific extras** from `research/recommendations.md` (average precision, balanced accuracy, validation-tuned threshold, pooled `val + test` counts, 5-fold stratified CV). Those use the probabilities saved by `predictions_file.py` and come after the per-field macro-F1 baseline.
- They do not yet enforce that the **test set is evaluated exactly once**. That guard comes before any test evaluation.
- They do not choose a winner. That is Hazem's decision, using the numbers in the run records.
- `mlflow` is optional and not used. The JSON run record is the source of truth.

**Questions or changes:** ask Aya. If you need a field that is not in the config, the predictions file or the run record, ask before adding your own format, so everyone keeps using the same one.